In [2]:
import pandas as pd
import psycopg2

# Configura tus datos de conexión
DB_HOST = '69.48.206.219'
DB_PORT = '5432'
DB_NAME = 'collection_db'
DB_USER = 'cobranza'
DB_PASS = 'cobranza2025'

# Leer el CSV
df = pd.read_csv('../data/investments_and_partners.csv')
# Asegura que los campos numéricos sean float
df['outstanding_capital'] = df['outstanding_capital'].astype(float)
df['investment_amount'] = df['investment_amount'].astype(float)
df['percentage'] = df['percentage'].astype(float)
df['partner_id'] = df['partner_id'].astype(int)

# Conexión a PostgreSQL
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASS
)
cur = conn.cursor()

# Diccionario para mapear external_investment_id a investment.id
investment_id_map = {}

# Insertar inversiones únicas
for ext_id, group in df.groupby('external_investment_id'):
    row = group.iloc[0]
    query = """
        INSERT INTO collection.investment
        (portafolio_origin, investment_date, outstanding_capital, investment_amount, portfolio_type)
        VALUES ('{portafolio_origin}', '{investment_date}', {outstanding_capital}, {investment_amount}, '{portfolio_type}')
        RETURNING id
    """.format(
        portafolio_origin=row['portafolio_origin'],
        investment_date=row['investment_date'],
        outstanding_capital=row['outstanding_capital'],
        investment_amount=row['investment_amount'],
        portfolio_type=row['portfolio_type']
    )
    
    cur.execute(query)
    
    investment_id = cur.fetchone()[0]
    investment_id_map[ext_id] = investment_id

# Insertar partner_investment
for _, row in df.iterrows():
    cur.execute("""
        INSERT INTO collection.partner_investment
        (partner_id, investment_id, percentage)
        VALUES (%s, %s, %s)
    """, (
        row['partner_id'],
        investment_id_map[row['external_investment_id']],
        row['percentage']
    ))

conn.commit()
cur.close()
conn.close()
print("Datos insertados correctamente.")

Datos insertados correctamente.
